In [8]:
!pip install -q -U openai-agents langchain-openai langchain-community chromadb pypdf streamlit google-genai pyngrok

In [9]:
!pip install -q streamlit google-genai pyngrok

In [10]:
!pip install -q -U openai

In [11]:
import os
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# Localiza os PDFs na pasta
caminhos_pdf = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]

if not caminhos_pdf:
    print("Nenhum arquivo .pdf encontrado na pasta.")
else:
    print(f"Criando Vector Store e enviando {len(caminhos_pdf)} arquivo(s) para a OpenAI...")

    vector_store = client.vector_stores.create(name="GoodWe_HCA_G2_KB")

    arquivos_abertos = [open(p, "rb") for p in caminhos_pdf]
    try:
        lote = client.vector_stores.file_batches.upload_and_poll(
            vector_store_id=vector_store.id,
            files=arquivos_abertos,
        )
    finally:
        for f in arquivos_abertos:
            f.close()

    print(f"Status do lote de upload: {lote.status} | arquivos: {lote.file_counts}")
    print("\n🎉 Vector Store pronta!")
    print("Copie EXATAMENTE a linha abaixo e cole no seu app.py:")
    print(f'VECTOR_STORE_ID = "{vector_store.id}"')

Nenhum arquivo .pdf encontrado na pasta.


In [12]:
%%writefile app.py
import os
import re
import asyncio
import streamlit as st
from dataclasses import dataclass
from typing import Union
from openai import OpenAI

# FRAMEWORK OPENAI AGENTS & GUARDRAILS
from agents import (
    SQLiteSession,
    Agent,
    Runner,
    RunContextWrapper,
    function_tool,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    TResponseInputItem,
    input_guardrail,
    output_guardrail,
)

st.set_page_config(page_title="GoodWe SmartCharge Assistant", layout="centered", page_icon="⚡")
st.title("⚡ GoodWe SmartCharge Assistant")
st.caption("Solução Inteligente para Infraestrutura de Recarga Elétrica (Vector Store na Nuvem)")

with st.sidebar:
    st.header("ChargeGrid Intelligence AI Assistant")
    st.info("Assistente Oficial GoodWe operando com base de dados persistente na nuvem da OpenAI.")
    st.markdown("---")
    st.markdown("**Tópicos Suportados:**\n- Controle de Demanda\n- Integração OCPP\n- Gestão de Tarifação\n- Interface e UX\n- Integração Solar")

api_key = os.environ.get("OPENAI_API_KEY")

# VECTOR STORE
VECTOR_STORE_ID = os.environ.get("VECTOR_STORE_ID", "")
client_openai = OpenAI(api_key=api_key) if api_key else None

if api_key:
    if VECTOR_STORE_ID and VECTOR_STORE_ID.startswith("vs_") and "SUBSTITUA" not in VECTOR_STORE_ID:
        st.sidebar.success("Base Conectada na Nuvem! (Vector Store ativa)")
    else:
        st.sidebar.warning("⚠️ Insira o VECTOR_STORE_ID gerado na etapa de upload.")

# CONTEXTO COMPARTILHADO DA EXECUÇÃO
@dataclass
class ContextoGoodWe:
    ultimo_contexto_rag: str = ""

# FERRAMENTA DE BUSCA INTELIGENTE
@function_tool
def buscar_manuais_goodwe(ctx: RunContextWrapper[ContextoGoodWe], duvida: str) -> str:
    """Ferramenta OBRIGATÓRIA. Consulta a base de manuais oficiais da GoodWe via file_search na Vector Store."""
    if not client_openai or not VECTOR_STORE_ID or "SUBSTITUA" in VECTOR_STORE_ID:
        return "Base de conhecimento indisponível."

    try:
        resposta = client_openai.responses.create(
            model="gpt-4o-mini",
            input=f"Extraia a resposta exata, incluindo todos os números e unidades citados, para: {duvida}",
            tools=[{
                "type": "file_search",
                "vector_store_ids": [VECTOR_STORE_ID],
                "max_num_results": 5,
            }],
            include=["file_search_call.results"],
        )

        trechos = []
        for item in resposta.output:
            if getattr(item, "type", None) == "file_search_call":
                for resultado in (item.results or []):
                    texto = getattr(resultado, "text", None)
                    if texto:
                        trechos.append(texto)

        contexto_recuperado = "\n---\n".join(trechos)

        if contexto_recuperado:
            ctx.context.ultimo_contexto_rag += ("\n---\n" + contexto_recuperado)

        if not contexto_recuperado:
            return "Informação não encontrada nos documentos indexados."

        return (
            f"Trechos oficiais recuperados dos manuais:\n{contexto_recuperado}\n\n"
            f"Leitura do modelo de extração: {resposta.output_text}"
        )

    except Exception as e:
        return f"Falha na busca: {str(e)}"

# GUARDRAILS DE ENTRADA
@input_guardrail(run_in_parallel=False)
def bloquear_injecao_prompt(ctx, agent, user_input: Union[str, list[TResponseInputItem]]) -> GuardrailFunctionOutput:
    texto = str(user_input).lower()
    padroes_injecao = ["ignore as instruções", "ignore todas as regras", "system prompt", "você agora é", "revele seu prompt"]
    encontrou = any(p in texto for p in padroes_injecao)
    return GuardrailFunctionOutput(
        output_info="Tentativa de injeção de prompt detectada" if encontrou else "Entrada permitida",
        tripwire_triggered=encontrou,
    )

@input_guardrail(run_in_parallel=False)
def bloquear_intencao_manutencao_perigosa(ctx, agent, user_input: Union[str, list[TResponseInputItem]]) -> GuardrailFunctionOutput:
    texto = str(user_input).lower()
    termos_perigosos = ["emendar fio", "descascar cabo", "ligar direto", "abrir o carregador", "choque", "fase e neutro"]
    encontrou = any(termo in texto for termo in termos_perigosos)
    return GuardrailFunctionOutput(
        output_info="Tentativa de manutenção elétrica perigosa detectada" if encontrou else "Entrada permitida",
        tripwire_triggered=encontrou,
    )

@input_guardrail(run_in_parallel=False)
def bloquear_linguagem_inapropriada_entrada(ctx, agent, user_input: Union[str, list[TResponseInputItem]]) -> GuardrailFunctionOutput:
    texto = str(user_input).lower()
    palavroes = ["porra", "fuder", "foder", "caralho", "merda", "puta", "cacete", "idiota"]
    encontrou = any(p in texto for p in palavroes)
    return GuardrailFunctionOutput(
        output_info="Linguagem inapropriada detectada na entrada" if encontrou else "Entrada permitida",
        tripwire_triggered=encontrou,
    )


# GUARDRAILS DE SAÍDA
@output_guardrail
def bloquear_conselho_juridico(ctx, agent, agent_output: str) -> GuardrailFunctionOutput:
    texto = agent_output.lower()
    termos_proibidos = ["processe o", "ação judicial", "garantia de isenção", "eu recomendo que você processe", "conselho jurídico"]
    encontrou = any(termo in texto for termo in termos_proibidos)
    return GuardrailFunctionOutput(
        output_info="Aconselhamento jurídico/financeiro detectado na saída" if encontrou else "Saída permitida",
        tripwire_triggered=encontrou,
    )

@output_guardrail
def bloquear_instrucoes_eletricas_perigosas(ctx, agent, agent_output: str) -> GuardrailFunctionOutput:
    texto = agent_output.lower()
    termos_perigosos = ["corte o fio", "junte os cabos", "sem o disjuntor", "abra a tampa frontal", "toque no conector"]
    encontrou = any(termo in texto for termo in termos_perigosos)
    return GuardrailFunctionOutput(
        output_info="Instrução elétrica perigosa gerada pelo modelo" if encontrou else "Saída permitida",
        tripwire_triggered=encontrou,
    )

@output_guardrail
def bloquear_vazamento_identidade(ctx, agent, agent_output: str) -> GuardrailFunctionOutput:
    encontrou = "DIRETRIZES DE SEGURANÇA" in agent_output or "Você é o GoodWe" in agent_output
    return GuardrailFunctionOutput(
        output_info="Vazamento do System Prompt detectado na saída" if encontrou else "Saída permitida",
        tripwire_triggered=encontrou,
    )

# Guardrail anti-"invenção de especificações"
_PADRAO_NUMERO = re.compile(
    r"\d+(?:[.,]\d+)?\s*(?:%|v|a|w|kw|kwh|hz|mm|cm|kg|°c|ºc)?",
    re.IGNORECASE,
)

def _extrair_numeros(texto: str) -> set[str]:
    if not texto:
        return set()
    numeros = set()
    for bruto in _PADRAO_NUMERO.findall(texto):
        normalizado = bruto.strip().lower().replace(" ", "").replace(",", ".")
        if any(c.isdigit() for c in normalizado):
            numeros.add(normalizado)
    return numeros

@output_guardrail
def bloquear_invencao_especificacoes(ctx: RunContextWrapper[ContextoGoodWe], agent, agent_output: str) -> GuardrailFunctionOutput:
    contexto_rag = getattr(ctx.context, "ultimo_contexto_rag", "") or ""

    if not contexto_rag.strip():
        return GuardrailFunctionOutput(
            output_info="Ferramenta de busca não foi acionada nesta resposta — verificação numérica não aplicável",
            tripwire_triggered=False,
        )

    numeros_resposta = _extrair_numeros(agent_output)
    numeros_contexto = _extrair_numeros(contexto_rag)
    numeros_nao_encontrados = numeros_resposta - numeros_contexto

    encontrou = bool(numeros_nao_encontrados)
    return GuardrailFunctionOutput(
        output_info=(
            f"Números citados na resposta mas ausentes no contexto recuperado dos documentos: {sorted(numeros_nao_encontrados)}"
            if encontrou else "Todos os números citados aparecem no contexto recuperado — saída permitida"
        ),
        tripwire_triggered=encontrou,
    )

guardrails_entrada_goodwe = [bloquear_injecao_prompt, bloquear_intencao_manutencao_perigosa, bloquear_linguagem_inapropriada_entrada]
guardrails_saida_goodwe = [bloquear_conselho_juridico, bloquear_instrucoes_eletricas_perigosas, bloquear_vazamento_identidade, bloquear_invencao_especificacoes]
ferramentas = [buscar_manuais_goodwe]

# DEFINIÇÃO DO AGENTE
agente_goodwe = Agent(
    name="Especialista_GoodWe",
    instructions="""Você é o GoodWe SmartCharge Assistant, especialista nos carregadores HCA G2.
    Siga este fluxo obrigatório:
    1. Para QUALQUER pergunta técnica sobre corrente, tensão, potência ou especificações, CHAME IMEDIATAMENTE A FERRAMENTA `buscar_manuais_goodwe`.
    2. Leia atentamente os trechos oficiais retornados pela ferramenta e cite APENAS valores de tensão, corrente e demais especificações que apareçam literalmente nesses trechos. NUNCA estime, arredonde de forma criativa ou complete um valor numérico que não esteja no trecho recuperado.
    3. Responda em português de forma técnica e direta.
    4. Se a ferramenta retornar "Informação não encontrada nos documentos indexados" ou os trechos recuperados não contiverem o dado pedido, diga explicitamente ao usuário que essa informação não foi localizada na documentação disponível e sugira reformular a pergunta ou contatar o suporte GoodWe. Dizer que não encontrou é sempre preferível a inventar um número.""",
    tools=ferramentas,
    input_guardrails=guardrails_entrada_goodwe,
    output_guardrails=guardrails_saida_goodwe,
    model="gpt-4o-mini",
)

async def executar_agente(pergunta, sessao):

    contexto = ContextoGoodWe()
    return await Runner.run(agente_goodwe, pergunta, session=sessao, context=contexto)

# INTERFACE E MEMÓRIA
if api_key:
    if "messages" not in st.session_state:
        st.session_state.messages = []

    if "agent_session" not in st.session_state:
        st.session_state.agent_session = SQLiteSession("sessao_goodwe_123")

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    if user_question := st.chat_input("Como posso ajudar com a solução da GoodWe hoje?"):
        with st.chat_message("user"):
            st.markdown(user_question)
        st.session_state.messages.append({"role": "user", "content": user_question})

        with st.chat_message("assistant"):
            with st.spinner("Consultando dados na nuvem e analisando segurança..."):
                try:
                    resultado = asyncio.run(executar_agente(user_question, st.session_state.agent_session))
                    resposta_final = resultado.final_output

                    st.markdown(resposta_final)
                    st.session_state.messages.append({"role": "assistant", "content": resposta_final})

                except InputGuardrailTripwireTriggered as e:
                    alerta = f"🚨 **Bloqueio de Segurança na Entrada:** Operação barrada pelo sistema.\n*Motivo:* {str(e)}"
                    st.error(alerta)
                    st.session_state.messages.append({"role": "assistant", "content": alerta})

                except OutputGuardrailTripwireTriggered as e:
                    alerta = f"🚨 **Bloqueio de Segurança na Saída:** Resposta bloqueada antes da exibição.\n*Motivo:* {str(e)}"
                    st.error(alerta)
                    st.session_state.messages.append({"role": "assistant", "content": alerta})

                except Exception as e:
                    st.error(f"Erro inesperado na execução: {e}")
else:
    st.error("Erro de configuração: OPENAI_API_KEY não encontrada no ambiente.")

Overwriting app.py


In [13]:
!nohup streamlit run app.py --server.port 8501 &

nohup: appending output to 'nohup.out'


In [14]:
import os
import time
from pyngrok import ngrok
from google.colab import userdata

try:
    openai_secret = userdata.get('OPENAI_API_KEY')
    ngrok_secret = userdata.get('NGROK_AUTH_TOKEN')
    vector_store_secret = userdata.get('VECTOR_STORE_ID')
    os.environ["OPENAI_API_KEY"] = openai_secret
    os.environ["VECTOR_STORE_ID"] = vector_store_secret

    print(" Limpando túneis travados...")
    ngrok.kill()

    ngrok.set_auth_token(ngrok_secret)

    print(" Reiniciando o Streamlit...")
    os.system("pkill streamlit")
    os.system("streamlit run app.py --server.port 8501 &")

    print(" Aguardando o Streamlit inicializar antes de abrir o túnel...")
    time.sleep(5)

    public_url = ngrok.connect(8501).public_url
    print("\n" + "="*60)
    print(" SEU CHATBOT COM RAG DE PDFs REAIS ESTÁ ONLINE!")
    print(f" CLIQUE NESTE LINK PARA ABRIR O SITE: {public_url}")
    print("="*60 + "\n")

except userdata.NotebookAccessError:
    print(" ERRO CRÍTICO: Ative o acesso aos Secrets no menu esquerdo.")
except Exception as e:
    print(f" Erro ao iniciar: {e}")

 Limpando túneis travados...
 Reiniciando o Streamlit...
 Aguardando o Streamlit inicializar antes de abrir o túnel...


 Erro ao iniciar: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://suds-curfew-derived.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}

